In [ ]:
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, interpolate
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})


In [ ]:
folderVec = [
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_dji9443/xy264_z120_cml_smag017/microphones/mic16R",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_dji9443/xy264_z120_cml_smag017_neq/microphones/mic16R",
]

labelVec = [
    r"Present LBM Cumulant VERY coarse",
    r"Present LBM Cumulant neq VERY coarse",

]

dxVec = [
    0.0006,
    0.0006,
]

rPhy = 0.12 # m

omegaRPMVec = [
    4800,
    4800,
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    "#A17808",
    '#FF5733',
    "#119100",
    "#B700FF",
    "#FA7BA1",
]

markerVec = [
    'o',
    '^',
    's',
    "D",
    '*',
]

rho0LB = 1.0
gamma = 1.0
rhoPhy = 1.204
unitConverterVec = [ uc(dx, rhoPhy, gamma) for dx in dxVec ]

refPressure = 2.0E-5 # Pa



In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

In [ ]:
def prepare(t, p, t_start=None, resample=True):
    t = np.asarray(t, float); p = np.asarray(p, float)
    if t_start is not None:
        m = t >= t_start
        t, p = t[m], p[m]
    if resample:
        dt = np.min(np.diff(t))
        tu = np.arange(t[0], t[-1], dt)
        p = interpolate.interp1d(t, p, kind='cubic')(tu)
        t = tu
    dt = np.mean(np.diff(t))
    p = p - p.mean()
    return t, p, 1.0 / dt


def welch_spl(p, fs, pRef=2e-5, nperseg=None, overlap=0.5, window='hann'):
    if nperseg is None:
        # nperseg = 2**int(np.log2(len(p)/8))      # 约分成 8~15 段平均
        nperseg = 2**int(np.log2(len(p)/1))
    f, Pxx = signal.welch(p, fs=fs, window=window, nperseg=nperseg,
                          noverlap=int(nperseg*overlap), scaling='density',
                        #   detrend='constant')
                          detrend='linear')
    df = f[1] - f[0]
    SPL_nb  = 10*np.log10(np.maximum(Pxx*df, 1e-30) / pRef**2)  # 每根谱线 SPL, dB
    SPL_psd = 10*np.log10(np.maximum(Pxx,   1e-30) / pRef**2)   # 谱密度级, dB/Hz
    return f, SPL_nb, SPL_psd, Pxx

def oaspl_time(p, pRef=2e-5):
    return 20 * np.log10(p.std() / pRef)

In [ ]:

avgStartFactor = 0.2
angleVecVec = []

freqVecVec = []
splVecVec = []
oasplVecVec = []

for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/microphone*.txt"))

    angleVec = []
    freqVec = []
    splVec = []
    oasplVec = []
    for mic in range(0, len(fileList)):
        x, y, z = get_probe_coords(fileList[mic])
        radians = np.arctan2(z, -x)
        degree = np.rad2deg(radians)
        # print(x, z, "{:.1f}".format(degree))
        angleVec.append(degree)
        stepCol = read_col_probe(fileList[mic], 0, 2)
        pressureCol = read_col_probe(fileList[mic], 2, 2)

        idxStart = int(avgStartFactor * len(stepCol))
        timeCol = unitConverterVec[case].step_to_time(stepCol[idxStart:])
        presCol = pressureCol[idxStart:]
        t, p, fs = prepare(timeCol, presCol, None, True)
        freqs, spl_nb, spl_psd, pxx = welch_spl(p, fs, refPressure)
        oaspl = oaspl_time(presCol)

        freqVec.append(freqs)
        splVec.append(spl_nb)
        oasplVec.append(oaspl)

    angleVecVec.append(angleVec)
    freqVecVec.append(freqVec)
    splVecVec.append(splVec)
    oasplVecVec.append(oasplVec)

# print(oasplVecVec)


In [ ]:

#! plot spl-frequency
idx = 0
figSpl, axSpl = plt.subplots(1, 1, figsize=(10, 6), facecolor='w', edgecolor='w')
for case in range(0, len(folderVec)):
    axSpl.semilogx(freqVecVec[case][idx], splVecVec[case][idx], c=colorVec[case], label=labelVec[case])

axSpl.set_ylim(0, None)
axSpl.set_xlim(100, 35000)
axSpl.set_xlabel("Frequency (Hz)")
axSpl.set_ylabel("SPL (dB)")
axSpl.legend(loc='lower left', frameon=False)
print(f"degree={angleVecVec[case][idx]}")


In [ ]:

#! plot angle-oaspl
figOaSpl, axOaSpl = plt.subplots(1, 1, figsize=(4, 6), facecolor='w', edgecolor='w')
for case in range(0, len(folderVec)):
    axOaSpl.plot(oasplVecVec[case], angleVecVec[case], marker=markerVec[case], ms=8, mew=1.5, fillstyle='none', c=colorVec[case], lw=0, label=labelVec[case])

axOaSpl.set_xlim(50, 80)
axOaSpl.set_xlabel("OASPL (dB)")
axOaSpl.set_ylabel(r"$\theta$ (deg)")

